# Geographic-Scale results viewer

Loads a scaling collection's `results.jsonl` plus the `.npz` sidecars in `arrays/` and renders:

- the cross-k summary plot (baseline ★, scaling mean ± std, random 1/(k+1) reference)
- the per-baseline quant-footprint and per-species tables
- a per-run drill-down (pick any row, recompute every per-species plot from the npz)

All recomputation — no retraining.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import os
import pyrootutils

os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '3')

ROOT = pyrootutils.setup_root(
    search_from=Path.cwd(),
    indicator='pyproject.toml',
    pythonpath=True,
    dotenv=True,
)

## Parameters

In [ ]:
COLLECTION = "paris_10"
BUILD_MODEL = 'sincnet'
NON_TARGET_NAME = 'non_target'

RESULTS_FILE = ROOT / 'models' / COLLECTION / BUILD_MODEL / 'results.jsonl'
ARRAYS_DIR  = RESULTS_FILE.parent / 'arrays'
print(f'jsonl : {RESULTS_FILE}  exists={RESULTS_FILE.exists()}')
print(f'arrays: {ARRAYS_DIR}  exists={ARRAYS_DIR.exists()}')

## Cross-k summary plot

In [ ]:
from building.scaling import summarize_results, plot_summary

baseline_metrics, summary_df = summarize_results(RESULTS_FILE)
print(f'Baseline recall   : {baseline_metrics.recall:.4f}')
print(f'Baseline precision: {baseline_metrics.precision:.4f}')
print(f'Baseline F1       : {baseline_metrics.f1:.4f}')
summary_df

In [ ]:
plot_summary(summary_df, baseline=baseline_metrics)

## Baselines table

In [ ]:
from building.scaling import print_baselines
from building.scaling.runner import load_results, BaselineRunResult, ScalingResult
import pandas as pd

catalog = None  # print_baselines wants a catalog; reuse load_dataset_catalog if needed.
rows = load_results(RESULTS_FILE)
baseline_rows = [r for r in rows if isinstance(r, BaselineRunResult)]
scaling_rows  = [r for r in rows if isinstance(r, ScalingResult)]
print(f'baselines: {len(baseline_rows)}   scaling: {len(scaling_rows)}')

if baseline_rows:
    stats_df = pd.DataFrame([
        {
            'target': r.target_class,
            'flash_kb': r.quant_stats.model_size_kb if r.quant_stats else None,
            'arena_kb': r.quant_stats.arena_size_kb if r.quant_stats else None,
            'mflops':   r.quant_stats.flops_mflops if r.quant_stats else None,
            'quant_loss': r.metrics.loss,
            'float_loss': r.float_metrics.loss if r.float_metrics else None,
            'quant_macro_targets_f2': (r.metrics.macro_targets.f2 if r.metrics.macro_targets else None),
            'float_macro_targets_f2': (r.float_metrics.macro_targets.f2 if (r.float_metrics and r.float_metrics.macro_targets) else None),
        }
        for r in baseline_rows
    ])
    display(stats_df.style.format(precision=4).set_caption('Baseline quant footprint + losses'))

## Per-baseline per-species metrics (quantized)

One row per (baseline_target, class). Quantized values at the run's saved threshold; `non_target` rows are kept for completeness but excluded from the macro summary.

In [ ]:
rows_pc = []
for r in baseline_rows:
    if r.metrics.per_class is None:
        continue
    for name, m in r.metrics.per_class.items():
        rows_pc.append({
            'baseline_target': r.target_class,
            'class': name,
            'support': m.support,
            'precision': m.precision,
            'recall': m.recall,
            'f1': m.f1,
            'f2': m.f2,
            'auc': m.auc,
        })
pc_df = pd.DataFrame(rows_pc)
if not pc_df.empty:
    display(pc_df.style.format(precision=4).set_caption('Per-class quantized metrics, per baseline run'))

## Per-run drill-down

Pick any row (baseline or scaling sample) and re-render the full per-species display block from its `.npz` sidecar.

In [ ]:
# Pick a row. For baseline: set the species. For scaling: set k + sample_idx.
PICK_BASELINE_TARGET: str | None = baseline_rows[0].target_class if baseline_rows else None
PICK_SCALING_K: int | None = None
PICK_SAMPLE_IDX: int | None = None

def _find_row():
    if PICK_BASELINE_TARGET is not None:
        for r in baseline_rows:
            if r.target_class == PICK_BASELINE_TARGET:
                return r
    if PICK_SCALING_K is not None and PICK_SAMPLE_IDX is not None:
        for r in scaling_rows:
            if r.k == PICK_SCALING_K and r.sample_idx == PICK_SAMPLE_IDX:
                return r
    return None

row = _find_row()
assert row is not None, 'No matching row — set PICK_* above.'
npz_path = Path(row.npz_path) if row.npz_path else None
print(f'picked: {row.run_type}  label_names={row.label_names}')
print(f'npz   : {npz_path}  exists={npz_path is not None and npz_path.exists()}')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from building.models import model_eval as M

assert npz_path is not None and npz_path.exists(), 'NPZ sidecar missing.'
npz = np.load(npz_path, allow_pickle=False)
y_true = npz['y_true']
y_score_float = npz['y_score_float']
y_score_quant = npz['y_score_quant']
label_names = list(row.label_names or [str(s) for s in npz['label_names']])
non_target_names = [NON_TARGET_NAME] if NON_TARGET_NAME in label_names else []

# Reuse the per-class thresholds saved with the run (defaults to 0.5 if missing).
per_class = (row.metrics.per_class or {})
thresholds = np.array(
    [per_class.get(n).threshold if (per_class.get(n) and per_class[n].threshold is not None) else 0.5
     for n in label_names],
    dtype=float,
)

float_eval = M.eval_result_from_predictions(
    backend='keras', label_names=label_names,
    y_true=y_true, y_score=y_score_float,
    threshold=thresholds, threshold_mode='fixed',
    non_target_names=non_target_names,
)
quant_eval = M.eval_result_from_predictions(
    backend='tflite', label_names=label_names,
    y_true=y_true, y_score=y_score_quant,
    threshold=thresholds, threshold_mode='fixed',
    non_target_names=non_target_names,
)
cmp = M.QuantComparison(float_eval=float_eval, quant_eval=quant_eval)
M.print_summary(quant_eval)

### Per-species table (quantized)

In [ ]:
tbl = M.per_class_table(quant_eval)
tbl_targets = tbl[~tbl['class'].isin(non_target_names)]
display(tbl_targets.style.format(precision=4))
if quant_eval.macro_f1_targets is not None:
    print(
        f'Macro (targets only) — precision={quant_eval.macro_precision_targets:.4f}  '
        f'recall={quant_eval.macro_recall_targets:.4f}  '
        f'F1={quant_eval.macro_f1_targets:.4f}  '
        f'F2={quant_eval.macro_f2_targets:.4f}  '
        f'AUC={quant_eval.macro_auc_targets:.4f}'
    )

### Metric-vs-threshold sweep (quantized)

In [ ]:
M.plot_metric_sweep_panel(quant_eval, target_only=True); plt.show()

### ROC per species (quantized)

In [ ]:
M.plot_roc(quant_eval, target_only=True); plt.show()
for name, mclass in quant_eval.per_class.items():
    if name in non_target_names:
        continue
    print(f'  {name:<28s} AUC={mclass.auc:.4f}' if mclass.auc is not None else f'  {name:<28s} n/a')

### Float vs INT8 comparison

In [ ]:
display(M.comparison_table(cmp).style.set_caption('Float vs INT8 per-class'))
M.plot_quantization_drop(cmp, metric='f2'); plt.show()
M.print_comparison(cmp)